In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import ttest_ind

sns.set(style="whitegrid")

In [ ]:
df = pd.read_csv("data/zomato.csv", encoding="latin1")
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Latitude": "lat",
    "Longitude": "lon",
    "Aggregate rating": "rating",
    "Votes": "votes",
    "Average Cost for two": "cost"
})
print(df.shape)
df.head()

In [9]:
cities = df["City"].dropna().unique()

g_cities = [c for c in cities if str(c).startswith("G")]

print(g_cities)

['Gainesville', 'Ghaziabad', 'Goa', 'Gurgaon', 'Guwahati']


In [10]:

city = "Ghaziabad"   # change as needed

df = df[df["City"] == city].copy()
df = df.dropna(subset=["lat", "lon"])

print("Total data points:", len(df))

Total data points: 25


In [12]:
# Convert to numeric safely
for col in ["votes", "rating", "cost"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing values (NO inplace)
df["votes"] = df["votes"].fillna(df["votes"].median())
df["rating"] = df["rating"].fillna(df["rating"].median())
df["cost"] = df["cost"].fillna(df["cost"].median())

# Drop missing coordinates
df = df.dropna(subset=["lat", "lon"])

# Remove outliers
def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    return df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]

for col in ["votes", "cost"]:
    df = remove_outliers(df, col)

print("After cleaning:", df.shape)

After cleaning: (22, 21)


In [ ]:
df["population"] = np.log1p(df["votes"])
df["traffic"] = df["rating"] * 20
df["rent"] = np.log1p(df["cost"])
df["competition"] = df.groupby(["City", "Locality"])["Locality"].transform("count")

def norm(x):
    return (x - x.min()) / (x.max() - x.min()) if x.max()!=x.min() else 0

# Normalize per city (IMPORTANT)
df["pop_n"] = df.groupby("City")["population"].transform(norm)
df["traffic_n"] = df.groupby("City")["traffic"].transform(norm)
df["rent_n"] = df.groupby("City")["rent"].transform(norm)
df["comp_n"] = df.groupby("City")["competition"].transform(norm)

# Final score
df["score"] = (
    0.3 * df["pop_n"] +
    0.3 * df["traffic_n"] -
    0.2 * df["rent_n"] -
    0.2 * df["comp_n"]
)

df.head()

In [ ]:
#Histogram
plt.figure(figsize=(6,4))
plt.hist(df["score"], bins=25)
plt.title("Score Distribution")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
#Correlation heatmap
cols = ["population", "traffic", "rent", "competition", "score"]

plt.figure(figsize=(6,4))
sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
#Clustering
features = df[["lat","lon","population","traffic","rent","competition"]]

scaler = MinMaxScaler()
X = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

df.head()

In [ ]:
#Best global location
cluster_df = df.groupby("cluster").agg({
    "lat":"mean",
    "lon":"mean",
    "score":"mean"
}).reset_index()

best = cluster_df.loc[cluster_df["score"].idxmax()]

print("🌍 Best Location (Global):")
print("Latitude:", round(best["lat"], 5))
print("Longitude:", round(best["lon"], 5))
print("Score:", round(best["score"], 3))

In [ ]:
#Hypothesis testing
high = df[df["traffic"] > df["traffic"].median()].copy()
low = df[df["traffic"] <= df["traffic"].median()].copy()

t_stat, p_val = ttest_ind(high["score"], low["score"])

print("T-statistic:", round(t_stat,4))
print("P-value:", round(p_val,5))

if p_val < 0.05:
    print("Result: Significant difference")
else:
    print("Result: Not significant")

In [ ]:
#Boxplot
high["group"] = "High Traffic"
low["group"] = "Low Traffic"

combined = pd.concat([high, low])

plt.figure(figsize=(6,4))
sns.boxplot(x="group", y="score", data=combined)
plt.title("Score Comparison (High vs Low Traffic)")
plt.show()

In [ ]:
#Denisty plot
plt.figure(figsize=(6,4))
sns.kdeplot(high["score"], label="High Traffic", fill=True)
sns.kdeplot(low["score"], label="Low Traffic", fill=True)
plt.legend()
plt.title("Score Density Distribution")
plt.show()

In [ ]:
#Mean comparison
means = [high["score"].mean(), low["score"].mean()]

plt.figure(figsize=(5,3))
plt.bar(["High Traffic","Low Traffic"], means)
plt.title("Mean Score Comparison")
plt.show()